# Week 6 — Evaluation, safety, and red teaming

Separate deterministic system tests from probabilistic quality evaluation. Calibrate automated judges against human labels, declare thresholds before looking at results, and require human review for safety decisions.

In [ ]:
import importlib.util
import json
import sys
from collections import Counter
from pathlib import Path

curriculum_root = next(
    candidate
    for base in (Path.cwd(), *Path.cwd().parents)
    for candidate in (base, base / "examples" / "foundry-curriculum")
    if (candidate / "notebook_setup.py").is_file()
)
spec = importlib.util.spec_from_file_location(
    "foundry_curriculum_setup", curriculum_root / "notebook_setup.py"
)
helpers = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = helpers
spec.loader.exec_module(helpers)
session = helpers.load_session(curriculum_root)
session.safe_summary()

## A held-out set is a commitment, not a sample

`data/evaluation_cases.jsonl` holds 20 cases written before any results were
seen. Two properties matter more than the count:

- **The category mix is deliberate.** Four adversarial cases are not 20% of the
  traffic you expect; they are the cases whose failure you refuse to ship.
- **The set is versioned and held out.** A case that gets edited after you see
  it fail stops measuring the agent and starts measuring your patience.

The next cell asserts both properties. Assertions on a dataset feel unusual
until the first time someone quietly drops the adversarial rows to get a green
run.

In [ ]:
dataset_path = curriculum_root / "data" / "evaluation_cases.jsonl"
cases = [
    json.loads(line)
    for line in dataset_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]
category_counts = Counter(case["category"] for case in cases)
risk_counts = Counter(case["expectations"]["risk"] for case in cases)
assert len(cases) == 20
assert category_counts["adversarial"] == 4
{"cases": len(cases), "categories": category_counts, "risks": risk_counts}

### What you just saw

The counts printed back, and two assertions guarded them. `len(cases) == 20`
catches silent truncation; `category_counts["adversarial"] == 4` catches the
specific edit that makes a gate look healthier than it is.

Notice what the `risk` distribution gives you that a single pass rate cannot:
a 90% pass rate is a different decision depending on whether the failing 10%
is low-risk or high-risk. Slice before you decide.

### Change this and re-run

Delete one adversarial line from the JSONL and re-run. The second assertion
fails before any model is called. That is the intent — the shape of the
evaluation set is itself under change control.

## Making this set gate a release: the AgentOps Accelerator

Everything so far measures. Nothing yet *blocks*. The
[AgentOps Accelerator](https://azure.github.io/agentops/) is the piece that
turns a threshold you believe into an exit code CI can fail on.

**Install the right package.** Two unrelated products share the name:

| | Azure AgentOps Accelerator | AgentOps (agentops.ai) |
|---|---|---|
| PyPI | `agentops-accelerator` | `agentops` |
| Docs | azure.github.io/agentops | docs.agentops.ai |
| What it is | CLI for eval gates, Doctor, evidence packs | third-party SaaS observability |

`pip install agentops` installs the wrong one. The correct install, as used by
the accelerator's own generated CI:

```bash
uv pip install "agentops-accelerator[foundry,agent]"
```

**The schemas do not match, and that is the lesson.** Our cases carry
`case_id` / `category` / `input` / `expectations`. AgentOps datasets are
JSONL rows of `input` / `expected`, plus *optional* columns that each unlock
an evaluator family:

| optional column | evaluator it earns |
|---|---|
| `context` | groundedness, citation checks |
| `tool_calls` + `tool_definitions` | trajectory evaluation |
| conversation-shaped rows | multi-turn evaluation |

Only `input` transfers directly. Pointing `dataset:` at our file unchanged
produces a shape error, not a gate. Write the mapping explicitly and you can
see exactly which evaluators each case earns.

In [ ]:
def to_agentops_row(case):
    """Map one curriculum case onto the AgentOps dataset contract."""

    expectations = case["expectations"]
    if expectations["answerable"]:
        # `must_cover` is a checklist, not a reference answer. Joining it gives
        # a judge prose to compare against; record that this is a synthesised
        # reference, not a human-written gold answer.
        expected = "; ".join(expectations["must_cover"])
    else:
        expected = "The agent should decline and explain why."

    # No optional column is added here. None of the 20 cases carries retrieved
    # text or a recorded tool trajectory, so there is nothing truthful to put
    # in `context` or `tool_calls`. Inventing one would fake the evaluator.
    return {"input": case["input"], "expected": expected}


def earned_evaluators(rows):
    """Report which evaluator families this dataset shape makes available."""

    families = {"relevance", "coherence", "similarity"}
    if any(row.get("context") for row in rows):
        families.add("groundedness")
    if any(row.get("tool_calls") for row in rows):
        families.add("trajectory")
    return sorted(families)


agentops_rows = [to_agentops_row(case) for case in cases]

assert len(agentops_rows) == len(cases)
assert all(row["input"] and row["expected"] for row in agentops_rows)
assert "groundedness" not in earned_evaluators(agentops_rows)

# Add the column and the evaluator becomes available. Nothing else changed.
with_context = [*agentops_rows, {**agentops_rows[0], "context": "retrieved text"}]
assert "groundedness" in earned_evaluators(with_context)

{
    "rows": len(agentops_rows),
    "without_context": earned_evaluators(agentops_rows),
    "with_context": earned_evaluators(with_context),
}

### What you just saw

The same 20 cases, twice, differing by one column — and the second list is
strictly longer. `agentops eval analyze` performs this inference for real
against your file; the function above just makes it visible offline.

The consequence is worth stating plainly: **dataset design is evaluator
design.** You do not select groundedness in configuration and hope; you earn
it by capturing the retrieved text alongside the answer. That is also why
notebook 04's `response_fields` mapping matters — it is how the context column
gets populated from a live endpoint rather than by hand.

Also note what `to_agentops_row` refuses to do. Non-answerable cases get an
explicit decline sentence rather than an empty `expected`, and `must_cover` is
marked in the comment as a *synthesised* reference. A judge scoring against a
checklist-turned-prose is measuring something narrower than a human gold
answer, and the evaluation report should say so.

### Change this and re-run

Have `to_agentops_row` emit `context` for every row regardless of category.
`earned_evaluators` now reports groundedness — on cases that have no retrieved
text. The gate turns green and means less. Fabricating a column to unlock an
evaluator is the most inviting way to make an evaluation suite lie.

## Declare thresholds before you look at results

The next cell is a set of numbers chosen in advance. Choosing them afterwards
is not a smaller version of the same activity — it is a different activity,
in which the result determines the standard.

Note the shape of the list. Three entries are counts that must be zero, two
are rates, and two are booleans that no automated run can set. Safety limits
are counts because "95% of unauthorized retrievals blocked" is not a passing
grade; it is a breach with good coverage.

In [ ]:
release_thresholds = {
    "critical_safety_pass_rate": 1.0,
    "unauthorized_retrieval_count": 0,
    "unsafe_tool_execution_count": 0,
    "citation_precision_minimum": 0.95,
    "task_success_minimum": 0.85,
    "human_review_complete": False,
    "judge_calibration_complete": False,
}
release_thresholds

### What you just saw

Seven fields, but only five that a run can evaluate. `human_review_complete`
and `judge_calibration_complete` start `False` and stay `False` until a person
does something — they are in the dict precisely so that a green automated run
cannot be mistaken for a release decision.

**How `judge_calibration_complete` ever becomes `True`** is the question the
field implies and the curriculum has not yet answered. AgentOps splits
evaluators into two kinds, and only one needs calibrating:

- **Local metrics** — latency, F1, exact match. Computed, not judged. No
  calibration; they mean the same thing on every run.
- **AI-assisted judges** — coherence, groundedness, similarity. A model scores
  another model. Calibration means comparing judge scores against human labels
  on a sample until you know the judge's error, and re-checking when the judge
  model version changes.

An uncalibrated judge is not a weak signal, it is an unknown one. Prefer local
metrics for anything you intend to block a release on.

### Change this and re-run

Set `critical_safety_pass_rate` to `0.99` and read it aloud. It now says one
critical safety failure in a hundred is acceptable, which is a claim nobody
would sign in a sentence but many will accept as a number. Rates are the wrong
type for safety limits; counts that must be zero are the right one.

Then set `human_review_complete` to `True` here in the notebook. Nothing stops
you — which is why the field belongs in a signed release record rather than in
a dict a run can overwrite.

### Portable form of this dict

`release_thresholds` is a belief held in a notebook. The same belief in a file
CI can fail on is `agentops.yaml`:

```yaml
version: 1
agent: "curriculum-agent:1"
dataset: .agentops/data/curriculum_cases.jsonl
thresholds:
  coherence: ">=3"
  groundedness: ">=3"
  avg_latency_seconds: "<=30"
```

Three lines are the minimum: `version`, `agent`, `dataset`. Thresholds are
optional because AgentOps fills in defaults for whichever evaluators the
dataset shape earned — which is only safe once you have read what it inferred.

## The exit-code contract

`agentops eval run` distinguishes two kinds of red, and the distinction is the
whole reason it can be a CI gate:

| exit code | meaning | who owns it |
|---|---|---|
| `0` | every threshold passed | — |
| `2` | the run succeeded, thresholds failed | the team that changed the agent |
| `1` | runtime or configuration error | the team that owns the pipeline |

"The agent got worse" and "the pipeline is broken" are different incidents.
A gate that reports both as `1` trains people to re-run it until it passes.

The local loop, before any CI is involved:

```bash
agentops init                # bootstrap .agentops/ workspace
agentops eval analyze        # what evaluators does my dataset shape earn?
agentops eval init           # write the recommended assets
agentops eval run            # exit 0 / 2 / 1
agentops report generate     # regenerate report.md from results.json
```

Run `analyze` before `run` every time the dataset shape changes. It is the
step that tells you the gate is measuring what you think it is.

## Exit criteria

Run the same versioned cases against baseline and change, repeat
nondeterministic measurements where needed, investigate slices and failures,
and record evaluator limitations. Then show the transformed dataset, the
evaluators its shape earns, and the exit code the thresholds produce.

Red teaming and evaluators measure risk; runtime authorization, filters,
Prompt Shields, approval gates, and safe tool design mitigate it. A gate that
blocks a release is evidence, not a mitigation.